## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-07-20-50-57 +0000,startribune,Gophers’ upset bid falls apart as unbeaten UCL...,https://www.startribune.com/minnesota-gophers-...
1,2026-11-07-20-48-00 +0000,startribune,Live: Blake leads Holy Angels 4-1 in day’s sec...,https://www.startribune.com/minnesota-mshsl-bo...
2,2026-11-07-20-41-00 +0000,wsj,Six Dead in Shooting at Market Square in Poland,https://www.wsj.com/world/europe/six-dead-in-s...
3,2026-11-07-20-39-58 +0000,nypost,Everyone in charge from sports to politics sho...,https://nypost.com/2026/11/07/sports/everyone-...
4,2026-11-07-20-36-05 +0000,nypost,Heartless crooks ruin California high school f...,https://nypost.com/2026/11/07/us-news/copper-w...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2685/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
53,trump,41
101,new,20
56,white,16
342,election,16
191,house,16


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
198,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...,204
258,2026-11-06-22-18-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...,154
38,2026-11-07-18-59-01 +0000,nypost,Trump offers UFC boss Dana White role of White...,https://nypost.com/2026/11/07/us-news/trump-of...,121
247,2026-11-06-22-53-56 +0000,wapo,Stephen Miller is named new White House chief ...,https://www.washingtonpost.com/politics/2026/1...,121
289,2026-11-06-21-24-00 +0000,wsj,Here’s five things to know about Stephen Mille...,https://www.wsj.com/politics/elections/five-th...,117


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
198,204,2026-11-07-02-00-00 +0000,wsj,President Trump has chosen Stephen Miller—an a...,https://www.wsj.com/politics/elections/stephen...
110,98,2026-11-07-13-00-05 +0000,chitri,Bill Foster vs Jeff Walter: How Illinois’ 12th...,https://www.chicagotribune.com/2026/11/07/bill...
83,58,2026-11-07-16-09-00 +0000,wsj,Is AI a white-collar job killer? Strong hiring...,https://www.wsj.com/economy/jobs/is-ai-taking-...
197,54,2026-11-07-02-00-00 +0000,wsj,The wave election that gave Democrats Congress...,https://www.wsj.com/politics/elections/2028-el...
258,47,2026-11-06-22-18-00 +0000,wsj,Trump said that longtime adviser Stephen Mille...,https://www.wsj.com/politics/elections/trump-n...
99,43,2026-11-07-14-30-07 +0000,nypost,Power-hungry DSA vows to ‘break US-Israel rela...,https://nypost.com/2026/11/07/us-news/dsa-vows...
174,41,2026-11-07-07-27-44 +0000,bbc,Meghan's father Thomas Markle under house arre...,https://www.bbc.co.uk/news/articles/cqgkvv8m8p...
85,39,2026-11-07-15-59-20 +0000,nypost,Historic California theme park makes heartbrea...,https://nypost.com/2026/11/07/business/six-fla...
18,37,2026-11-07-20-02-01 +0000,wapo,"Israeli president taps ex-general Eisenkot, op...",https://www.washingtonpost.com/world/2026/11/0...
190,37,2026-11-07-03-00-00 +0000,wsj,Elon Musk-run companies like SpaceX and Tesla ...,https://www.wsj.com/politics/policy/elon-musk-...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
